In [1]:
generated_insights = """
Based on the provided ydata-profiling report, here is a summary of the data quality issues and recommended actions:

*   **Missing Values**
    *   *Columns:* `Student_Name`, `House`, `Subject`, `Class_Date`, `Score`, `Attendance_Status`, `Teacher`
    *   *Issue:* Multiple columns contain missing values (e.g., 6 missing values in `Subject`, 4 in `Class_Date`, 3 in `Score`, 2 in `Student_Name`, `House`, and `Teacher`, and 1 in `Attendance_Status`).
    *   *Recommended Action:* Impute missing values where appropriate or drop the affected rows/columns depending on analysis requirements.

*   **Duplicate Rows**
    *   *Columns:* Entire dataset
    *   *Issue:* The dataset contains 6 duplicate rows (1.19%).
    *   *Recommended Action:* Remove the duplicate rows.

*   **Incorrect Data Types / Formatting**
    *   *Columns:* `Score`, `Class_Date`, `Year_Level`
    *   *Issue:* 
        *   `Score` is read as text, contains formatting anomalies (e.g., trailing `.0.0`), and invalid values (e.g., `999`, `105`, `120`).
        *   `Class_Date` uses mixed formats (e.g., dashes vs. slashes like `1992/02/14`) and contains text/invalid entries (e.g., `tbd`, `recorded not`).
        *   `Year_Level` contains anomalous characters/values (e.g., `-`, `Y`, `year`).
    *   *Recommended Action:* Clean and cast `Score` to a numeric data type, standardize date formats and clean invalid entries in `Class_Date`, and filter or correct invalid characters in `Year_Level`.

*   **Inconsistent Categorical Values**
    *   *Columns:* `Student_Name`, `Subject`, `Attendance_Status`, `Teacher`, `House`, `Year_Level`
    *   *Issue:* 
        *   `Student_Name` contains leading whitespace (e.g., `" Viktor Malfoy"`).
        *   `Subject` contains case inconsistencies (`potions` vs `Potions`) and variations like trailing spaces (`"Herbology "`) or abbreviations (`dada` vs `Defence Against the Dark Arts`).
        *   `Attendance_Status` contains inconsistent casing/values (e.g., `here` alongside `Present`).
        *   `Teacher` has mixed casing (`snape` vs `Snape`, `McGonagall`, `Flitwick`).
        *   `House` and `Year_Level` contain out-of-domain or anomalous categorical strings (evident from high distinct counts/lengths).
    *   *Recommended Action:* Trim whitespace, standardize text casing (e.g., lowercase/title-case), harmonize abbreviations and synonyms, and correct invalid category entries.
    """

In [9]:
KEYWORDS = {
    "missing_values": ["missing", "null", "nan", "empty"],
    "duplicates": ["duplicate", "repeated"],
    # add more as you write more skill files, e.g.
    # "outliers": ["outlier", "extreme"],
    # "dtype_fixes": ["dtype", "data type", "convert", "to datetime"],
}


def pick_skills(insights):

    """insights: The string output of model generated cleaning insights."""
    
    text = insights.lower()
    chosen = []
    for skill_name, words in KEYWORDS.items():
        if any(w in text for w in words):
            chosen.append(skill_name)
    return chosen

In [10]:
chosen_keywords = pick_skills(generated_insights)

print(chosen_keywords)

['missing_values', 'duplicates']


In [13]:
from pathlib import Path

SKILLS_DIR = Path("skills")      # a folder named "skills" next to your notebook

def load_skill_text(skill_names):
    
    """Reads each chosen skill file and joins them together."""
    
    parts = []
    for name in skill_names:
        path = SKILLS_DIR / f"{name}.md"
        if path.exists():
            parts.append(path.read_text(encoding="utf-8"))
    return "\n\n---\n\n".join(parts)

In [15]:
loaded_skill_text = load_skill_text(chosen_keywords)

loaded_skill_text

'# Skill: missing_values\n\n## When to use\nAn insight mentions missing values, nulls, NaN, or empty cells in a column.\n\n## Rules\n- Numeric column, low skew: fill with the median.\n- Numeric column, more than 40% missing: drop the column instead of filling.\n- Categorical column: fill with the mode, or "Unknown" if there is no clear mode.\n- Always check that the column exists before touching it.\n- Work on `df` directly and never reload the file.\n\n## Example\n```python\ncol = "age"\nif col in df.columns:\n    if df[col].isna().mean() > 0.40:\n        df = df.drop(columns=[col])\n    else:\n        df[col] = df[col].fillna(df[col].median())\n```\n\n\n---\n\n# Skill: duplicates\n\n## When to use\nAn insight mentions duplicate rows or repeated records.\n\n## Rules\n- Drop exact duplicate rows only, keep the first occurrence.\n- Reset the index afterwards.\n- Print how many rows were removed.\n\n## Example\n```python\nbefore = len(df)\ndf = df.drop_duplicates(keep="first").reset_inde